# HMS v8 — does the boundary shortcut also affect a pretrained EEG foundation model? (CBraMod)

**Inputs:** (1) HMS competition data; (2) `notebookbb6726eb71` (v5 output: `results_v5`); (3) `notebook86c832275b` **Version 5** (`results_v3` predictions). To resume after the time budget, also attach this notebook's own output.
**Settings:** GPU T4, **Internet ON** (downloads the pretrained weights from Hugging Face, ~20 MB). **Save Version → Save & Run All**.

**Runs (5 folds each, stage 1, same folds/labels/recipe length as our models):**
1. `orig` 50 s — the standard clean benchmark setting.
2. `orig` 200 s — zero-padded windows, as anyone would fine-tune a foundation model on longer HMS windows (boundary visible).
3. `ba_noreweight` 200 s — our boundary-agnostic recipe (mirror-fill + random truncation).

**Tests:** the same shortcut signatures as for our models (gain on EEGs with no real context, reflect and truncate interventions, truncation probability shift) and the real-context effect after the fix. If CBraMod shows the shortcut too, the finding is not specific to our architectures.

In [1]:
import os, glob, json, math, time, random, shutil
import numpy as np
import pandas as pd

CFG = dict(
    DATA_ROOT=None,                       # None = auto-detect under /kaggle/input
    CACHE_DIR="/kaggle/working/cache",    # replaced in the v2 settings cell below
    RESULTS_DIR="/kaggle/working/results",
    FS=100,                               # target sampling rate (Hz)
    CTX_SEC=50,
    CACHE_CTX_SEC=100,
    JITTER_SEC=1.0,                       # train-time random shift of the window centre
    PATCH=25,                             # samples per token (0.25 s at 100 Hz)
    N_FOLDS=5, FOLDS=[0], SEED=42,
    MODELS=["mamba", "transformer", "gru", "cnn"],
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    EPOCHS=15, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    TRAIN_MIN_VOTES=0,
    HIGH_VOTE=10,                         # threshold for the high-agreement evaluation subset
    NUM_WORKERS=4, AMP=True,
    USE_OFFICIAL_MAMBA2=False,
    RUN_BENCHMARK=True,
    BENCH_CTX=[50, 100, 200, 400, 600, 1200],
)

In [2]:
import pyarrow.parquet as pq
import multiprocessing as mp
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
MP = mp.get_context("fork")   # Kaggle is Linux; fork shares globals with workers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")

ROOT = find_root()
print("data:", ROOT)

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Labels and patient-level folds

In [3]:
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
P_COLS = [f"p_{c}" for c in CLASSES]
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0   # experts labelled the central 10 s of each 50 s window

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k
assert df.groupby("patient_id")["fold"].nunique().max() == 1, "patient leaked across folds"

print(f"rows={len(df):,}  eegs={df.eeg_id.nunique():,}  patients={df.patient_id.nunique():,}")
print(df.groupby("fold").agg(rows=("eeg_id", "size"), eegs=("eeg_id", "nunique"), patients=("patient_id", "nunique")))
print(df.drop_duplicates("eeg_id")["expert_consensus"].value_counts())

rows=106,800  eegs=17,089  patients=1,950
       rows  eegs  patients
fold                       
0     17085  2795       383
1     25502  3689       386
2     21174  3585       394
3     22141  3238       397
4     20898  3782       390
expert_consensus
Other      7196
Seizure    2785
LPD        2577
GRDA       1828
GPD        1787
LRDA        916
Name: count, dtype: int64


## Preprocessing
18-channel bipolar double banana + midline (identical to I-CARE's 18 bipolar pairs). Band-pass 0.5–40 Hz at 200 Hz, then every second sample (100 Hz; the 40 Hz low-pass prevents aliasing). Amplitudes stay in µV (no per-window z-scoring), because absolute voltage — suppression, high-amplitude discharges — is clinically informative.

In [4]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),    # left temporal chain
           ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),    # right temporal chain
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),    # left parasagittal chain
           ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),    # right parasagittal chain
           ("Fz", "Cz"), ("Cz", "Pz")]                                 # midline
FLIP_PERM = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11, 16, 17]   # left <-> right hemisphere
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH = len(BIPOLAR)
SRC_FS = 200
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")
CLIP_UV = 500.0

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def _nrows(eid):
    return pq.ParquetFile(f"{ROOT}/train_eegs/{eid}.parquet").metadata.num_rows

def plan_cache(df, half_sec):
    g = df.groupby("eeg_id")["center"].agg(["min", "max"]).reset_index()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        n = pool.map(_nrows, g["eeg_id"].tolist(), chunksize=64)
    g["n_src"] = n
    s0 = np.floor((g["min"] - half_sec) * SRC_FS).clip(lower=0).astype(np.int64)
    s0 -= s0 % 2
    s1 = np.minimum(g["n_src"], np.ceil((g["max"] + half_sec) * SRC_FS)).astype(np.int64)
    g["s0"], g["length"] = s0, ((s1 - s0 + 1) // 2).clip(lower=0)
    g["offset"] = np.concatenate([[0], np.cumsum(g["length"].to_numpy())[:-1]])
    return g[["eeg_id", "n_src", "s0", "length", "offset"]]

def _build_one(args):
    eid, s0, length = args
    x = load_bipolar(f"{ROOT}/train_eegs/{eid}.parquet")[s0::2][:length]
    out = np.zeros((length, N_CH), np.float16)
    out[:len(x)] = x
    return out

def build_cache(df):
    meta_path = f"{CFG['CACHE_DIR']}/meta.json"
    want = dict(cache_ctx=CFG["CACHE_CTX_SEC"], jitter=CFG["JITTER_SEC"], fs=CFG["FS"], n_eegs=int(df.eeg_id.nunique()))
    if os.path.exists(meta_path) and json.load(open(meta_path)) == want:
        print("cache found, reusing")
        return pd.read_csv(f"{CFG['CACHE_DIR']}/index.csv")
    half = CFG["CACHE_CTX_SEC"] / 2 + CFG["JITTER_SEC"]
    idx = plan_cache(df, half)
    total = int(idx["length"].sum())
    need = total * N_CH * 2
    free = shutil.disk_usage(CFG["CACHE_DIR"]).free
    print(f"cache: {total/CFG['FS']/3600:,.1f} h of EEG, {need/1e9:.2f} GB needed, {free/1e9:.1f} GB free")
    if need > 0.9 * free:
        raise RuntimeError("Not enough disk: lower CACHE_CTX_SEC or point CACHE_DIR at a larger disk")
    mm = np.lib.format.open_memmap(f"{CFG['CACHE_DIR']}/signals.npy", mode="w+", dtype=np.float16, shape=(total, N_CH))
    jobs = list(zip(idx["eeg_id"], idx["s0"], idx["length"]))
    t0 = time.time()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        for i, (x, off) in enumerate(zip(pool.imap(_build_one, jobs, chunksize=16), idx["offset"])):
            mm[off:off + len(x)] = x
            if i % 2000 == 0:
                print(f"  {i:,}/{len(jobs):,}  {time.time()-t0:.0f}s")
    mm.flush(); del mm
    idx.to_csv(f"{CFG['CACHE_DIR']}/index.csv", index=False)
    json.dump(want, open(meta_path, "w"))
    print(f"cache built in {time.time()-t0:.0f}s")
    return idx

## Dataset
Each item is a `CTX_SEC` window centred on the labelled 10 s. Positions outside the recording are zero-filled and flagged in a mask; the SSM treats masked steps as identity transitions (Δ = 0), the Transformer ignores them via key-padding, the readout never pools them.
Training samples one random sub-window per EEG per epoch, so long recordings with many overlapping sub-windows don't dominate. Validation uses the middle sub-window of each EEG (the competition test set also has one row per EEG).

In [5]:
class HMSDataset(Dataset):
    """Self-contained (no notebook globals in __getitem__), so it works with any worker start method."""
    def __init__(self, rows, ctx_sec, train):
        assert ctx_sec <= CFG["CACHE_CTX_SEC"], "CTX_SEC exceeds the cached context; rebuild the cache"
        self.eid = rows["eeg_id"].to_numpy()
        self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.y = rows["y"].to_numpy()
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs)
        assert self.n % CFG["PATCH"] == 0
        self.train, self.jitter, self.flip = train, CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"
        self.mm = None

    def __len__(self):
        return len(self.eid)

    def __getitem__(self, i):
        if self.mm is None:                      # open lazily so each worker gets its own handle
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i]
        if self.train:
            c += np.random.uniform(-self.jitter, self.jitter)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        x = np.zeros((self.n, self.n_ch), np.float32)
        m = np.zeros(self.n, np.float32)
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b > a:
            x[a - i0:b - i0] = self.mm[off + a:off + b]
            m[a - i0:b - i0] = 1.0
        x = x.T / 50.0                           # ~unit scale while keeping absolute amplitude
        if self.train:
            if np.random.rand() < 0.5:
                x = x[self.flip]
            x = x * np.random.uniform(0.9, 1.1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(m), torch.from_numpy(self.p[i]), int(self.y[i])

class OnePerEEG(Sampler):
    def __init__(self, eids, seed):
        order = np.argsort(eids, kind="stable")
        _, starts = np.unique(eids[order], return_index=True)
        self.groups = np.split(order, starts[1:])
        self.seed, self.epoch = seed, 0
    def __len__(self):
        return len(self.groups)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = np.array([g[rng.integers(len(g))] for g in self.groups])
        rng.shuffle(idx)
        return iter(idx.tolist())

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]

## Models
All models: `Stem` (conv + 0.25 s patching → tokens) → sequence mixer → readout (pool over the central 10 s ⊕ pool over the full context) → 6 logits. Only the mixer differs.

The SSM is a bidirectional Mamba-2 block. `ssd()` is the chunked state-space-duality algorithm (exact); `ssd_reference()` is the plain recurrence used to verify it. `Mamba2.step()` is the recurrent form used for streaming.

In [6]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## CBraMod encoder (foundation model)
The encoder below reproduces braindecode's `CBraMod` (BSD-3-Clause, © braindecode developers; Wang et al., ICLR 2025) module-for-module, so the pretrained checkpoint `braindecode/cbramod-pretrained` loads with identical parameter names. It is vendored to avoid installing braindecode (which may pull a different torch on Kaggle).

Fine-tuning follows the original CBraMod recipe: the pretraining reconstruction layer `proj_out` is replaced by identity, and a task head reads the encoder output. Input: 200 Hz, 1 s patches, amplitude in units of 100 µV. Our pipeline stores 100 Hz signals band-passed at 0.5–40 Hz in units of 50 µV, so the wrapper multiplies by 0.5 and upsamples ×2 (no information lost: the signal has no content above 40 Hz).

The wrapper ignores the mask on purpose: a user fine-tuning a foundation model zero-pads windows that run past the recording, which is exactly the condition under which the boundary shortcut is available. The boundary-agnostic variant mirror-fills instead.

In [7]:
import copy

class CrissCrossLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.self_attn_s = nn.MultiheadAttention(d_model // 2, nhead // 2, dropout=dropout, batch_first=True)
        self.self_attn_t = nn.MultiheadAttention(d_model // 2, nhead // 2, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward); self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout1, self.dropout2 = nn.Dropout(dropout), nn.Dropout(dropout)
        self.activation = nn.GELU()
    def forward(self, x):                                   # norm_first, as in CBraMod
        x = x + self._sa_block(self.norm1(x))
        return x + self.dropout2(self.linear2(self.dropout(self.activation(self.linear1(self.norm2(x))))))
    def _sa_block(self, x):
        bz, ch, pn, d = x.shape
        xs = x[..., : d // 2].permute(0, 2, 1, 3).reshape(bz * pn, ch, -1)
        xt = x[..., d // 2:].reshape(bz * ch, pn, -1)
        xs = self.self_attn_s(xs, xs, xs, need_weights=False)[0].reshape(bz, pn, ch, -1).permute(0, 2, 1, 3)
        xt = self.self_attn_t(xt, xt, xt, need_weights=False)[0].reshape(bz, ch, pn, -1)
        return self.dropout1(torch.cat((xs, xt), dim=3))

class LayerStack(nn.Module):
    def __init__(self, layer, n):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n)])
    def forward(self, x):
        for mod in self.layers:
            x = run_block(mod, x)                           # gradient checkpointing if CFG["GRAD_CKPT"]
        return x

class PatchEmbedding(nn.Module):
    CKSPN = ((25, 49, 25, 24, (5, 25)), (25, 3, 1, 1, (5, 25)), (25, 3, 1, 1, (5, 25)))
    def __init__(self, patch_size=200, drop_prob=0.1):
        super().__init__()
        self.patch_size = patch_size
        p = patch_size
        for _, k, s, pad, _ in self.CKSPN:
            p = int((p + 2 * pad - k) / s + 1)
        self.d_model = self.CKSPN[-1][0] * p
        self.positional_encoding = nn.Sequential(nn.Conv2d(self.d_model, self.d_model, kernel_size=(19, 7), stride=(1, 1), padding=(9, 3), groups=self.d_model))
        self.mask_encoding = nn.Parameter(torch.zeros(patch_size), requires_grad=False)
        layers, last = [], 1
        for c, k, s, pad, norm in self.CKSPN:
            layers += [nn.Conv2d(last, c, kernel_size=(1, k), stride=(1, s), padding=(0, pad)), nn.GroupNorm(*norm), nn.GELU()]
            last = c
        self.proj_in = nn.Sequential(*layers)
        self.spectral_proj = nn.Sequential(nn.Linear(patch_size // 2 + 1, self.d_model), nn.Dropout(drop_prob))
    def forward(self, x):
        bz, ch, pn, ps = x.shape
        mx = x.reshape(bz, 1, ch * pn, ps)
        emb = self.proj_in(mx)
        d, p2 = emb.shape[1], emb.shape[3]
        emb = emb.reshape(bz, d, ch, pn, p2).permute(0, 2, 3, 1, 4).reshape(bz, ch, pn, d * p2)
        spec = torch.fft.rfft(mx.reshape(-1, ps).float(), dim=-1, norm="forward").abs().to(emb.dtype).reshape(bz, ch, pn, -1)
        emb = emb + self.spectral_proj(spec)
        return emb + self.positional_encoding(emb.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)

class CBraModEncoder(nn.Module):
    def __init__(self, patch_size=200, dim_feedforward=800, n_layer=12, nhead=8, emb_dim=200, drop_prob=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embedding = PatchEmbedding(patch_size, drop_prob)
        d = self.patch_embedding.d_model
        self.encoder = LayerStack(CrissCrossLayer(d, nhead, dim_feedforward, drop_prob), n_layer)
        self.proj_out = nn.Sequential(nn.Linear(d, emb_dim))   # pretraining reconstruction layer (replaced by identity for fine-tuning)
        self.d_model = d
    def forward(self, x):                                   # x: (B, C, T) at 200 Hz, T divisible by patch_size
        B, C, Tn = x.shape
        return self.proj_out(self.encoder(self.patch_embedding(x.reshape(B, C, Tn // self.patch_size, self.patch_size))))

def load_cbramod_weights(enc):
    """Download the pretrained checkpoint and load it; returns the fraction of encoder parameters that were loaded."""
    from huggingface_hub import hf_hub_download, list_repo_files
    files = list_repo_files(CFG["FM_HUB"])
    name = next((f for f in ("model.safetensors", "pytorch_model.bin") if f in files), None)
    name = name or next(f for f in files if f.endswith((".safetensors", ".bin", ".pth", ".pt")))
    fp = hf_hub_download(CFG["FM_HUB"], name)
    if fp.endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(fp)
    else:
        sd = torch.load(fp, map_location="cpu", weights_only=False)
        sd = sd.get("state_dict", sd.get("model", sd)) if isinstance(sd, dict) else sd
    own = enc.state_dict()
    def strip(k):
        for pre in ("model.", "module.", "backbone.", "encoder_model."):
            if k.startswith(pre):
                return k[len(pre):]
        return k
    sd = {strip(k): v for k, v in sd.items()}
    ok = {k: v for k, v in sd.items() if k in own and own[k].shape == v.shape}
    enc.load_state_dict(ok, strict=False)
    n_own = sum(v.numel() for k, v in own.items() if not k.startswith("proj_out"))
    n_ok = sum(v.numel() for k, v in ok.items() if not k.startswith("proj_out"))
    missing = [k for k in own if k not in ok and not k.startswith("proj_out")]
    print(f"CBraMod checkpoint {CFG['FM_HUB']}/{name}: loaded {len(ok)}/{len(own)} tensors, {n_ok / n_own:.1%} of encoder parameters; "
          f"unused checkpoint tensors: {len(sd) - len(ok)}; missing: {missing[:5]}")
    return n_ok / n_own

class FMNet(nn.Module):
    """CBraMod encoder + the same readout as our models (pool over the central 10 s ⊕ pool over the full window)."""
    def __init__(self, pretrained=True):
        super().__init__()
        self.enc = CBraModEncoder(drop_prob=CFG["FM_DROPOUT"])
        if pretrained:
            frac = load_cbramod_weights(self.enc)
            assert frac > 0.95, "pretrained CBraMod weights did not load — check internet (Settings → Internet on) and the log above"
        self.enc.proj_out = nn.Identity()
        d = self.enc.d_model
        self.head = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(CFG["FM_DROPOUT"]), nn.Linear(2 * d, len(CLASSES)))
    def forward(self, x, m=None):                           # x: (B, 18, T) at 100 Hz in units of 50 µV
        x = F.interpolate(x.float() * 0.5, scale_factor=2, mode="linear", align_corners=False)
        h = self.enc(x).mean(1)                             # (B, patches, d): average over channels
        P = h.shape[1]; c0 = P // 2 - 5
        z = torch.cat([h[:, c0:c0 + 10].mean(1), h.mean(1)], -1)
        return self.head(z)

## v8 settings

In [8]:
CFG.update(dict(
    CACHE_CTX_SEC=200, CACHE_DIR="/tmp/hms_cache", RESULTS_DIR="/kaggle/working/results_v8",
    PREV_GLOBS=["/kaggle/input/**/results_v8/*.json", "/kaggle/input/**/results_v8/*_valpred.npy"],   # resume across sessions
    PRED_GLOBS=["/kaggle/input/**/results_v3/*_valpred.npy", "/kaggle/input/**/results_v5/*_valpred.npy"],
    CTX=200, N_FOLDS=5, EPOCHS=6, BATCH=64, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    FM_HUB="braindecode/cbramod-pretrained", FM_LR=1e-4, FM_HEAD_LR=1e-3, FM_DROPOUT=0.1,
    FM_TOKENS_PER_MICRO=16 * 18 * 200,          # (channels x 1 s patches) per micro-batch; halved automatically on out-of-memory
    FM_PRETRAINED=True,
    DROPOUT=0.1, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False,
    P_TRUNC=0.5, W_CLIP=(0.25, 4.0),
    RUNS=[("orig", 50), ("orig", 200), ("ba_noreweight", 200)],     # (variant, context seconds), in priority order
    EST_MIN_PER_FOLD={50: 8, 200: 25},
    TIME_BUDGET_H=9.0,
    B=4000, MIN_N=20, T_GRID=list(np.round(np.linspace(0.5, 3.0, 51), 3)),
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
))
CFG_OVERRIDES = {"TIME_BUDGET_H": 11.0, "EST_MIN_PER_FOLD": {50: 18, 200: 95}}
CFG.update(CFG_OVERRIDES)
T_START = time.time()
os.makedirs(CFG["CACHE_DIR"], exist_ok=True); os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)

n_copied = 0
for pat in CFG["PREV_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        dst = os.path.join(CFG["RESULTS_DIR"], os.path.basename(fp))
        if not os.path.exists(dst):
            shutil.copy(fp, dst); n_copied += 1
print(f"resumed {n_copied} files from earlier v8 outputs")
PREDS = {}
for pat in CFG["PRED_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        PREDS.setdefault(os.path.basename(fp), fp)
print(f"found {len(PREDS)} earlier prediction files (Mamba references)")

resumed 30 files from earlier v8 outputs
found 160 earlier prediction files (Mamba references)


## Per-EEG and per-row context availability

In [9]:
cache_idx = build_cache(df)
CACHE = {int(r.eeg_id): (int(r.offset), int(r.s0), int(r.length)) for r in cache_idx.itertuples()}
DUR = dict(zip(cache_idx["eeg_id"].astype(int), cache_idx["n_src"] / SRC_FS))

def extra_context(center, dur, ctx=CFG["CTX"]):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(dur - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(dur - center, 25), 0, None)

df["dur_s"] = df["eeg_id"].map(DUR)
df["extra_s"] = extra_context(df["center"].to_numpy(), df["dur_s"].to_numpy())
df["ctx_bin"] = pd.cut(df["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)

VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)
T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); PID = OOF["patient_id"].to_numpy(); FOLD = OOF["fold"].to_numpy()
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(NV >= lo) & (NV <= hi), "vote_group"] = name
OOF["label"] = np.array(CLASSES)[OOF["y"].to_numpy()]
VG, CB = OOF["vote_group"].to_numpy(), OOF["ctx_bin"].to_numpy()
print(pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).to_string())

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)

def load_oof(model, ctx, stage):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(f"{model}_ctx{ctx}_f{f}_{stage}_valpred.npy")
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

UP, INV = np.unique(PID, return_inverse=True)
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)

def boot(d, msk):
    if msk.sum() == 0:
        return (float("nan"),) * 4
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (W @ s_) / (W @ n_)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return float(d[msk].mean()), float("nan"), float("nan"), float("nan")
    lo, hi = np.percentile(r, [2.5, 97.5])
    return float(d[msk].mean()), lo, hi, max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    idx = np.argsort(np.where(ok, p, np.inf)); k = ok.sum(); run = 0.0
    for i_, j in enumerate(idx[:k]):
        run = max(run, min(1.0, (k - i_) * p[j])); out[j] = run
    return out

SUBSETS = {"all EEGs": np.ones(len(OOF), bool)}
for b in CFG["CTX_LABELS"]:
    SUBSETS[f"all | {b}"] = CB == b
for g in ["1-2 votes", "3-9 votes", ">=10 votes"]:
    SUBSETS[f"{g} | all"] = VG == g
    for b in CFG["CTX_LABELS"]:
        SUBSETS[f"{g} | {b}"] = (VG == g) & (CB == b)

cache: 380.3 h of EEG, 4.93 GB needed, 1038.2 GB free
  0/17,089  1s
  2,000/17,089  21s
  4,000/17,089  43s
  6,000/17,089  64s
  8,000/17,089  85s
  10,000/17,089  107s
  12,000/17,089  128s
  14,000/17,089  149s
  16,000/17,089  170s
cache built in 183s
ctx_bin     full (>=140 s)  none (<10 s)  partial (10-140 s)    All
vote_group                                                         
1-2 votes               25          4526                 296   4847
3-9 votes              297          2298                4047   6642
>=10 votes             169          2648                2783   5600
All                    491          9472                7126  17089


## B. Boundary-agnostic (BA) training of 200 s models
Three changes make the recording boundary invisible or useless to the model:
1. **Reflect-fill, no mask:** the part of the 200 s window outside the recording is filled by mirroring the real signal, and the mask says "real" everywhere — so padding can no longer reveal where the recording ends.
2. **Random truncation (p = 0.5):** during training, part of the real context on each side is hidden and mirror-filled too, so mirrored regions also appear in long recordings and stop being a "short recording" signature. The central 50 s is never hidden.
3. **Label balancing across context availability (variant `ba`):** each training row is weighted by Σ_k t_k · π_k / π_{k | context bin}, removing the incentive to infer the label from how much context exists. `ba_noreweight` drops this step (ablation).

Evaluation uses reflect-fill without truncation (`ba`) and, for the residual-shortcut test, the central 50 s only, mirror-filled (`trunc50`).

In [10]:
class BADataset(Dataset):
    def __init__(self, rows, ctx_sec, train, mode="ba", weights=None):
        self.eid = rows["eeg_id"].to_numpy(); self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32); self.y = rows["y"].to_numpy()
        self.w = np.ones(len(rows), np.float32) if weights is None else np.asarray(weights, np.float32)
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs); self.train, self.mode = train, mode
        self.jitter, self.p_trunc, self.flip = CFG["JITTER_SEC"], CFG["P_TRUNC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"; self.mm = None
    def __len__(self):
        return len(self.eid)
    def window(self, i, rng):
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i] + (rng.uniform(-self.jitter, self.jitter) if self.train else 0.0)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        a, b = max(i0, 0), min(i0 + self.n, length)
        lo, hi = a - i0, b - i0                               # real signal inside the window: [lo, hi)
        cidx, h50 = self.n // 2, 25 * self.fs
        if self.mode == "trunc50":
            lo, hi = max(lo, cidx - h50), min(hi, cidx + h50)
        elif self.train and rng.random() < self.p_trunc:
            lo = int(rng.integers(lo, max(lo, cidx - h50) + 1))
            hi = int(rng.integers(min(hi, cidx + h50), hi + 1))
        if hi <= lo:
            return np.zeros((self.n, self.n_ch), np.float32), lo, hi
        real = np.asarray(self.mm[off + i0 + lo:off + i0 + hi], dtype=np.float32)
        x = np.pad(real, ((lo, self.n - hi), (0, 0)), mode="symmetric") if (lo > 0 or hi < self.n) else real
        return x, lo, hi
    def __getitem__(self, i):
        if self.mm is None:
            self.mm = np.load(self.path, mmap_mode="r")
        rng = np.random.default_rng()
        x, lo, hi = self.window(i, rng)
        m = np.ones(self.n, np.float32) if hi > lo else np.zeros(self.n, np.float32)
        x = x.T / 50.0
        if self.train:
            if rng.random() < 0.5:
                x = x[self.flip]
            x = x * rng.uniform(0.9, 1.1)
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), torch.from_numpy(m),
                torch.from_numpy(self.p[i]), torch.tensor(self.w[i]))

# sanity: the central 50 s is always real signal, the mask never reveals the boundary
_ds = BADataset(OOF.head(200), CFG["CTX"], train=True)
_ds.mm = np.load(_ds.path, mmap_mode="r")
_rng = np.random.default_rng(0)
for i in range(min(200, len(_ds))):
    _, lo, hi = _ds.window(i, _rng)
    tol = int(CFG["JITTER_SEC"] * CFG["FS"]) + 1                      # jitter can push the label edge by up to 1 s
    assert lo <= _ds.n // 2 - 25 * CFG["FS"] + tol and hi >= _ds.n // 2 + 25 * CFG["FS"] - tol, (i, lo, hi)
print("BA sanity check passed: central 50 s always real; mask uniform")

def balance_weights(rows):
    """w = sum_k t_k * pi_k / pi_{k|bin}: removes label information carried by context availability."""
    Tr = rows[P_COLS].to_numpy(np.float64)
    pi = Tr.mean(0)
    w = np.zeros(len(rows))
    for b in rows["ctx_bin"].unique():
        msk = (rows["ctx_bin"] == b).to_numpy()
        pib = np.clip(Tr[msk].mean(0), 1e-4, None)
        w[msk] = (Tr[msk] * (pi / pib)).sum(1)
    w = np.clip(w, *CFG["W_CLIP"])
    return w / w.mean()

def param_groups(model):
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim < 2 or n_.endswith(("A_log", "D", "dt_bias")) else decay).append(p)
    return [dict(params=decay, weight_decay=CFG["WD"]), dict(params=no_decay, weight_decay=0.0)]

def loader(ds, batch, sampler=None, drop_last=False):
    return DataLoader(ds, batch, sampler=sampler, shuffle=False, drop_last=drop_last, num_workers=CFG["NUM_WORKERS"],
                      pin_memory=DEV.type == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0,
                      multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, m, _, _ in dl:
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
            out.append(torch.softmax(model(x.to(DEV), m.to(DEV)).float(), -1).cpu())
    return torch.cat(out).numpy()

BA sanity check passed: central 50 s always real; mask uniform


## Evaluation views and fine-tuning
`EvalDataset` builds the original-model views: `standard` (zeros outside the recording), `reflect` (boundary hidden by mirror-fill), `truncate` (only the central 50 s, zeros elsewhere). The boundary-agnostic model is scored with `BADataset` views `ba` and `trunc50`.

In [11]:
class EvalDataset(Dataset):
    def __init__(self, rows, ctx_sec, mode):
        self.eid = rows["eeg_id"].to_numpy(); self.center = rows["center"].to_numpy(np.float64)
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs); self.mode = mode
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"; self.mm = None
    def __len__(self):
        return len(self.eid)
    def __getitem__(self, i):
        if self.mm is None:
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        i0 = int(round((self.center[i] - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        a, b = max(i0, 0), min(i0 + self.n, length)
        x = np.zeros((self.n, self.n_ch), np.float32); m = np.zeros(self.n, np.float32)
        if b > a:
            real = np.asarray(self.mm[off + a:off + b], dtype=np.float32)
            if self.mode == "reflect" and (a - i0 > 0 or i0 + self.n - b > 0):
                x = np.pad(real, ((a - i0, i0 + self.n - b), (0, 0)), mode="symmetric"); m[:] = 1.0
            else:
                x[a - i0:b - i0] = real; m[a - i0:b - i0] = 1.0
        if self.mode == "truncate":
            keep = np.zeros(self.n, bool); keep[self.n // 2 - 25 * self.fs:self.n // 2 + 25 * self.fs] = True
            x[~keep] = 0.0; m[~keep] = 0.0
        return (torch.from_numpy(np.ascontiguousarray((x * m[:, None]).T / 50.0)), torch.from_numpy(m),
                torch.zeros(len(CLASSES)), torch.tensor(1.0))

def fm_param_groups(model):
    groups = {}
    for n_, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lr = CFG["FM_HEAD_LR"] if n_.startswith("head") else CFG["FM_LR"]
        wd = 0.0 if p.ndim < 2 else CFG["WD"]
        groups.setdefault((lr, wd), []).append(p)
    return [dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()]

_MICRO = {}
def auto_micro(ctx):
    """Largest micro-batch (≤ BATCH) whose forward+backward fits in GPU memory at this context."""
    if ctx in _MICRO:
        return _MICRO[ctx]
    micro = max(1, min(CFG["BATCH"], CFG["FM_TOKENS_PER_MICRO"] // (N_CH * ctx)))
    if DEV.type == "cuda":
        mdl = FMNet(pretrained=False).to(DEV).train()
        while micro > 1:
            ok = True
            try:
                x = torch.randn(micro, N_CH, ctx * CFG["FS"], device=DEV)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=CFG["AMP"]):
                    out = mdl(x)
                out.float().sum().backward()
            except torch.cuda.OutOfMemoryError:
                ok = False
            x = out = None; mdl.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            if ok:
                micro = max(1, int(micro * 0.85))          # headroom for optimizer state and data
                break
            micro //= 2
        del mdl; torch.cuda.empty_cache()
    _MICRO[ctx] = micro
    print(f"micro-batch at {ctx} s: {micro}")
    return micro

def fm_tag(variant, ctx, fold):
    return f"cbramod_{variant}_ctx{ctx}_f{fold}"

EVAL_MODES = {"orig": {50: ["standard"], 200: ["standard", "reflect", "truncate"]}, "ba_noreweight": {200: ["ba", "trunc50"]}}

def eval_ds(variant, ctx, fold, mode):
    return BADataset(VA[fold], ctx, train=False, mode=mode) if variant.startswith("ba") else EvalDataset(VA[fold], ctx, mode)

def train_fm(variant, ctx, fold):
    tag = fm_tag(variant, ctx, fold)
    seed_all(CFG["SEED"] + fold)
    tr = df[df.fold != fold].reset_index(drop=True)
    micro = auto_micro(ctx); accum = math.ceil(CFG["BATCH"] / micro)
    ds = BADataset(tr, ctx, train=True) if variant.startswith("ba") else HMSDataset(tr, ctx, True)
    dl = loader(ds, micro, OnePerEEG(tr["eeg_id"].to_numpy(), CFG["SEED"] + fold), drop_last=True)
    model = FMNet(pretrained=CFG["FM_PRETRAINED"]).to(DEV)
    opt = torch.optim.AdamW(fm_param_groups(model))
    base_lrs = [g["lr"] for g in opt.param_groups]
    total = max(1, CFG["EPOCHS"] * (len(dl) // accum)); warm = max(1, int(CFG["WARMUP_FRAC"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    use_amp = CFG["AMP"] and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"\n=== {tag} | params {n_params(model):,} | micro {micro} x accum {accum} | lrs {sorted(set(base_lrs))}")
    t_run, hist = time.time(), []
    for ep in range(CFG["EPOCHS"]):
        model.train(); t0 = time.time(); tl = nb = 0.0
        opt.zero_grad(set_to_none=True)
        for i, (x, m, p, _) in enumerate(dl):
            x, m, p = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, m)
            loss = F.kl_div(F.log_softmax(logits.float(), -1), p, reduction="batchmean")
            scaler.scale(loss / accum).backward()
            tl += loss.item(); nb += 1
            if (i + 1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), sec=time.time() - t0))
        print(f"  {tag} ep {ep} | train KL {tl/max(nb,1):.4f} | {time.time()-t0:.0f}s")
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"{CFG['RESULTS_DIR']}/{tag}.pt")
    for mode in EVAL_MODES[variant][ctx]:
        P = predict(model, loader(eval_ds(variant, ctx, fold, mode), micro * 2))
        np.save(f"{CFG['RESULTS_DIR']}/{tag}_{mode}_valpred.npy", P)
    json.dump(dict(variant=variant, ctx=ctx, fold=fold, micro=micro, minutes=(time.time() - t_run) / 60, history=hist),
              open(f"{CFG['RESULTS_DIR']}/{tag}.json", "w"), indent=1)
    print(f"=== {tag} done in {(time.time()-t_run)/60:.1f} min")
    del model
    if DEV.type == "cuda": torch.cuda.empty_cache()

# check the pretrained weights load before spending GPU time
if CFG["FM_PRETRAINED"]:
    _m = FMNet(pretrained=True); del _m

def fm_done(variant, ctx, fold):
    return all(os.path.exists(f"{CFG['RESULTS_DIR']}/{fm_tag(variant, ctx, fold)}_{m}_valpred.npy") for m in EVAL_MODES[variant][ctx])

PLAN = [(v_, c_, f_) for v_, c_ in CFG["RUNS"] for f_ in range(CFG["N_FOLDS"])]
todo = [r for r in PLAN if not fm_done(*r)]
print(f"{len(PLAN)} runs planned, {len(PLAN) - len(todo)} already done, ~{sum(CFG['EST_MIN_PER_FOLD'][c_] for _, c_, _ in todo) / 60:.1f} GPU-h left (estimate)")
for v_, c_, f_ in todo:
    used = (time.time() - T_START) / 3600
    if used + CFG["EST_MIN_PER_FOLD"][c_] / 60 > CFG["TIME_BUDGET_H"]:
        print(f"time budget: stopping before {fm_tag(v_, c_, f_)} ({used:.2f} h used). Attach this output and rerun to continue.")
        break
    train_fm(v_, c_, f_)

model.safetensors: reconstructing file:   0%|          |  0.00B / 19.7MB            

model.safetensors: downloading bytes:           |  0.00B            

CBraMod checkpoint braindecode/cbramod-pretrained/model.safetensors: loaded 211/211 tensors, 100.0% of encoder parameters; unused checkpoint tensors: 0; missing: []
15 runs planned, 10 already done, ~7.9 GPU-h left (estimate)
micro-batch at 200 s: 13
CBraMod checkpoint braindecode/cbramod-pretrained/model.safetensors: loaded 211/211 tensors, 100.0% of encoder parameters; unused checkpoint tensors: 0; missing: []

=== cbramod_ba_noreweight_ctx200_f0 | params 4,886,806 | micro 13 x accum 5 | lrs [0.0001, 0.001]


/tmp/ipykernel_24/1459609724.py:102: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


  cbramod_ba_noreweight_ctx200_f0 ep 0 | train KL 1.0638 | 909s
  cbramod_ba_noreweight_ctx200_f0 ep 1 | train KL 0.7640 | 911s
  cbramod_ba_noreweight_ctx200_f0 ep 2 | train KL 0.6572 | 910s
  cbramod_ba_noreweight_ctx200_f0 ep 3 | train KL 0.5965 | 910s
  cbramod_ba_noreweight_ctx200_f0 ep 4 | train KL 0.5503 | 910s
  cbramod_ba_noreweight_ctx200_f0 ep 5 | train KL 0.5290 | 910s
=== cbramod_ba_noreweight_ctx200_f0 done in 93.1 min
CBraMod checkpoint braindecode/cbramod-pretrained/model.safetensors: loaded 211/211 tensors, 100.0% of encoder parameters; unused checkpoint tensors: 0; missing: []

=== cbramod_ba_noreweight_ctx200_f1 | params 4,886,806 | micro 13 x accum 5 | lrs [0.0001, 0.001]


/tmp/ipykernel_24/1459609724.py:102: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


... (3 outputs left out)

/tmp/ipykernel_24/1459609724.py:102: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


  cbramod_ba_noreweight_ctx200_f3 ep 0 | train KL 1.0005 | 883s
  cbramod_ba_noreweight_ctx200_f3 ep 1 | train KL 0.7342 | 883s
  cbramod_ba_noreweight_ctx200_f3 ep 2 | train KL 0.6505 | 882s
  cbramod_ba_noreweight_ctx200_f3 ep 3 | train KL 0.5894 | 883s
  cbramod_ba_noreweight_ctx200_f3 ep 4 | train KL 0.5369 | 881s
  cbramod_ba_noreweight_ctx200_f3 ep 5 | train KL 0.5149 | 882s
=== cbramod_ba_noreweight_ctx200_f3 done in 90.7 min
CBraMod checkpoint braindecode/cbramod-pretrained/model.safetensors: loaded 211/211 tensors, 100.0% of encoder parameters; unused checkpoint tensors: 0; missing: []

=== cbramod_ba_noreweight_ctx200_f4 | params 4,886,806 | micro 13 x accum 5 | lrs [0.0001, 0.001]


/tmp/ipykernel_24/1459609724.py:102: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)


  cbramod_ba_noreweight_ctx200_f4 ep 0 | train KL 1.0383 | 848s
  cbramod_ba_noreweight_ctx200_f4 ep 1 | train KL 0.7663 | 848s
  cbramod_ba_noreweight_ctx200_f4 ep 2 | train KL 0.6350 | 848s
  cbramod_ba_noreweight_ctx200_f4 ep 3 | train KL 0.5801 | 847s
  cbramod_ba_noreweight_ctx200_f4 ep 4 | train KL 0.5331 | 847s
  cbramod_ba_noreweight_ctx200_f4 ep 5 | train KL 0.5130 | 847s
=== cbramod_ba_noreweight_ctx200_f4 done in 87.6 min


## Results
Pooled out-of-fold over all EEGs, cross-fitted temperature (fitted on the other 4 folds), patient-cluster bootstrap. Mamba references come from the v3/v5 outputs.

In [12]:
def load_fm(variant, ctx, mode):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = f"{CFG['RESULTS_DIR']}/{fm_tag(variant, ctx, f)}_{mode}_valpred.npy"
        if not os.path.exists(fp):
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def load_named(fmt):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(fmt.format(f=f))
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def temper(P, t):
    L_ = np.log(np.clip(P, 1e-12, 1)) / t; L_ -= L_.max(1, keepdims=True); E = np.exp(L_)
    return E / E.sum(1, keepdims=True)

def fit_T(P):
    """cross-fitted temperatures (one per held-out fold), fitted on all EEGs of the other folds"""
    Ts = {}
    for f in range(CFG["N_FOLDS"]):
        tr = FOLD != f
        Ts[f] = min(CFG["T_GRID"], key=lambda t: (T[tr] * (np.log(T[tr] + 1e-15) - np.log(temper(P[tr], t) + 1e-15))).sum(1).mean())
    return Ts

def apply_T(P, Ts):
    out = np.empty_like(P)
    for f, t in Ts.items():
        out[FOLD == f] = temper(P[FOLD == f], t)
    return out

SETS = {"CBraMod 50 s": ("orig", 50, "standard"), "CBraMod 200 s original": ("orig", 200, "standard"),
        "CBraMod 200 s original | reflect": ("orig", 200, "reflect"), "CBraMod 200 s original | truncate": ("orig", 200, "truncate"),
        "CBraMod 200 s BA": ("ba_noreweight", 200, "ba"), "CBraMod 200 s BA | trunc50": ("ba_noreweight", 200, "trunc50")}
REFS = {"Mamba 50 s": "mamba_ctx50_f{f}_s1_valpred.npy", "Mamba 200 s original": "mamba_ctx200_f{f}_s1_valpred.npy",
        "Mamba 200 s BA": "mamba_ba_noreweight_ctx200_f{f}_ba_valpred.npy", "Mamba 200 s BA | trunc50": "mamba_ba_noreweight_ctx200_f{f}_trunc50_valpred.npy"}
RAW = {k: load_fm(*v) for k, v in SETS.items()}
RAW.update({k: load_named(v) for k, v in REFS.items()})
RAW = {k: v for k, v in RAW.items() if v is not None}
print("available:", list(RAW))

# a model's temperature is fitted on its own standard view and reused for its intervention views
BASE = {"CBraMod 200 s original | reflect": "CBraMod 200 s original", "CBraMod 200 s original | truncate": "CBraMod 200 s original",
        "CBraMod 200 s BA | trunc50": "CBraMod 200 s BA", "Mamba 200 s BA | trunc50": "Mamba 200 s BA"}
TS = {k: fit_T(v) for k, v in RAW.items() if k not in BASE}
CAL = {k: apply_T(v, TS[BASE.get(k, k)]) for k, v in RAW.items() if BASE.get(k, k) in TS}
KLR = {k: per_eeg_kl(v) for k, v in RAW.items()}
KLC = {k: per_eeg_kl(v) for k, v in CAL.items()}

rows = []
for k in RAW:
    if k in KLC:
        rows.append(dict(model=k, kl_raw_all=KLR[k].mean(), kl_cal_all=KLC[k].mean(), kl_cal_ge10=KLC[k][NV >= 10].mean(),
                         **{f"kl_cal | {b}": (KLC[k][CB == b].mean() if (CB == b).any() else np.nan) for b in CFG["CTX_LABELS"]}))
main = pd.DataFrame(rows)
main.to_csv(f"{CFG['RESULTS_DIR']}/v8_main.csv", index=False)
print("### Calibrated pooled OOF KL (intervention views use the temperature of the model's own view)\n" + main.round(4).to_string(index=False))

available: ['CBraMod 50 s', 'CBraMod 200 s original', 'CBraMod 200 s original | reflect', 'CBraMod 200 s original | truncate', 'CBraMod 200 s BA', 'CBraMod 200 s BA | trunc50', 'Mamba 50 s', 'Mamba 200 s original', 'Mamba 200 s BA', 'Mamba 200 s BA | trunc50']
### Calibrated pooled OOF KL (intervention views use the temperature of the model's own view)
                            model  kl_raw_all  kl_cal_all  kl_cal_ge10  kl_cal | none (<10 s)  kl_cal | partial (10-140 s)  kl_cal | full (>=140 s)
                     CBraMod 50 s      0.8465      0.7912       0.6367                 0.7553                       0.8362                   0.8297
           CBraMod 200 s original      0.7291      0.6983       0.5527                 0.6667                       0.7399                   0.7027
 CBraMod 200 s original | reflect      0.8870      0.8504       0.6644                 0.8753                       0.8275                   0.7052
CBraMod 200 s original | truncate      0.8754      0.

### Shortcut signatures for CBraMod (and the same tests for Mamba where available)
Negative difference = first view better.
1. **No-context gain:** 200 s vs 50 s on EEGs with no real extra context. A shortcut-free model cannot gain there.
2. **Reflect:** hiding the boundary of the original 200 s model by mirror-fill. A shortcut model gets worse.
3. **Truncate vs 50 s model** on full-context EEGs: the original 200 s model shown only the central 50 s, compared with a model trained at 50 s on the same signal.
4. **Real context effect after the fix:** BA 200 s vs the same model limited to the central 50 s, on EEGs that have real context.

In [13]:
def test(a, b, msk, label):
    if a not in KLC or b not in KLC or msk.sum() < CFG["MIN_N"]:
        return None
    m, lo, hi, p = boot(KLC[a] - KLC[b], msk)
    return dict(test=f"{a.split()[0]} | {label}", first=a, second=b, n=int(msk.sum()), kl_first=KLC[a][msk].mean(), kl_second=KLC[b][msk].mean(),
                diff=m, ci_lo=lo, ci_hi=hi, rel_pct=100 * m / KLC[b][msk].mean(), p=p)

NONE, FULL, REAL = CB == CFG["CTX_LABELS"][0], CB == CFG["CTX_LABELS"][2], CB != CFG["CTX_LABELS"][0]
ALL = np.ones(len(OOF), bool)
tests = []
for fam in ("CBraMod", "Mamba"):
    tests += [test(f"{fam} 200 s original", f"{fam} 50 s", NONE, "1 no-context gain, original"),
              test(f"{fam} 200 s BA", f"{fam} 50 s", NONE, "1 no-context gain, BA"),
              test(f"{fam} 200 s original | reflect", f"{fam} 200 s original", ALL, "2 reflect intervention"),
              test(f"{fam} 200 s original | truncate", f"{fam} 50 s", FULL, "3 truncated 200 s model vs 50 s model, full context"),
              test(f"{fam} 200 s BA", f"{fam} 200 s BA | trunc50", REAL, "4 real context effect after fix, EEGs with context"),
              test(f"{fam} 200 s BA", f"{fam} 200 s BA | trunc50", REAL & (VG == "3-9 votes"), "4 real context effect, 3-9 votes"),
              test(f"{fam} 200 s BA", f"{fam} 200 s BA | trunc50", REAL & (VG == ">=10 votes"), "4 real context effect, >=10 votes")]
tests += [test("CBraMod 50 s", "Mamba 50 s", ALL, "architecture, 50 s"),
          test("CBraMod 200 s BA", "Mamba 200 s BA", ALL, "architecture, 200 s BA"),
          test("CBraMod 50 s", "Mamba 50 s", NV >= 10, "architecture, 50 s, >=10 votes"),
          test("CBraMod 200 s BA", "Mamba 200 s BA", NV >= 10, "architecture, 200 s BA, >=10 votes")]
sig = pd.DataFrame([t for t in tests if t is not None])
if len(sig):
    sig["p_holm"] = holm(sig["p"].to_numpy())
    sig.to_csv(f"{CFG['RESULTS_DIR']}/v8_shortcut_tests.csv", index=False)
    print(sig.drop(columns=["first", "second"]).round(4).to_string(index=False))

                                                         test     n  kl_first  kl_second    diff   ci_lo   ci_hi  rel_pct      p  p_holm
                        CBraMod | 1 no-context gain, original  9472    0.6667     0.7553 -0.0886 -0.1201 -0.0585 -11.7316 0.0002  0.0040
                              CBraMod | 1 no-context gain, BA  9472    0.6729     0.7553 -0.0825 -0.1042 -0.0605 -10.9160 0.0002  0.0040
                             CBraMod | 2 reflect intervention 17089    0.8504     0.6983  0.1522  0.1290  0.1759  21.7930 0.0002  0.0040
CBraMod | 3 truncated 200 s model vs 50 s model, full context   491    0.9775     0.8297  0.1478  0.0814  0.2257  17.8146 0.0002  0.0040
 CBraMod | 4 real context effect after fix, EEGs with context  7617    0.7228     0.8038 -0.0811 -0.0951 -0.0679 -10.0888 0.0002  0.0040
                   CBraMod | 4 real context effect, 3-9 votes  4344    0.8283     0.9732 -0.1449 -0.1665 -0.1237 -14.8891 0.0002  0.0040
                  CBraMod | 4 real contex

In [14]:
shift = []
for a, b, lab in [("CBraMod 200 s original | truncate", "CBraMod 200 s original", "CBraMod original: truncate - standard"),
                  ("CBraMod 200 s BA | trunc50", "CBraMod 200 s BA", "CBraMod BA: trunc50 - ba"),
                  ("Mamba 200 s BA | trunc50", "Mamba 200 s BA", "Mamba BA: trunc50 - ba")]:
    if a in RAW and b in RAW and FULL.sum() >= CFG["MIN_N"]:
        shift.append(dict(comparison=lab, n_full=int(FULL.sum()), **{c: (RAW[a][FULL, k] - RAW[b][FULL, k]).mean() for k, c in enumerate(CLASSES)}))
if shift:
    print("### Mean probability shift on full-context EEGs when only the central 50 s is shown (Mamba original for reference: Seizure -0.21, Other +0.24)\n"
          + pd.DataFrame(shift).round(3).to_string(index=False))
print("outputs:", len(os.listdir(CFG["RESULTS_DIR"])), "files in", CFG["RESULTS_DIR"])

### Mean probability shift on full-context EEGs when only the central 50 s is shown (Mamba original for reference: Seizure -0.21, Other +0.24)
                           comparison  n_full  Seizure   LPD    GPD  LRDA   GRDA  Other
CBraMod original: truncate - standard     491   -0.121 0.093  0.004 0.023 -0.099  0.099
             CBraMod BA: trunc50 - ba     491   -0.058 0.030 -0.001 0.010 -0.007  0.027
               Mamba BA: trunc50 - ba     491   -0.016 0.003 -0.002 0.002 -0.005  0.018
outputs: 52 files in /kaggle/working/results_v8
